# SQLite: تکرار، اصلاحیه و بازیابی

داده‌های این نمونه ساختگی هستند؛ قیمت واقعی بازار نیستند.

## سلول ۱ — ورود کتابخانه‌ها

In [ ]:
from pathlib import Path
from datetime import datetime, UTC
import tempfile
import json
from copy import deepcopy
from importlib.resources import files
import httpx
import pandas as pd
from pydara import tse
from pydara.tse.samples import sample_transport

## سلول ۲ — تعریف توابع

In [ ]:
def revised_transport(state):
    base = sample_transport()
    def handle(request):
        if "GetClosingPriceDailyList/" in request.url.path:
            body = json.loads(files("pydara.tse.samples").joinpath("history_fixture.json").read_text())
            if state["corrected"]:
                body["closingPriceDaily"][0]["pClosing"] = 128
            n = int(request.url.path.rsplit("/", 1)[1])
            if n:
                body["closingPriceDaily"] = sorted(body["closingPriceDaily"], key=lambda r:r["dEven"], reverse=True)[:n]
            return httpx.Response(200, json=body)
        return base.handle_request(request)
    return httpx.MockTransport(handle)

## سلول ۳ — تنظیمات

In [ ]:
SYMBOL = 'فولاد'
OUTPUT = Path(tempfile.mkdtemp(prefix="pydara-part3-store-"))
state = {"corrected": False}
config = tse.ClientConfig(network=tse.NetworkConfig(requests_per_second=1000000, max_retries=0))

## سلول ۴ — اجرا

In [ ]:
with tse.Client(config=config, data_dir=OUTPUT, transport=revised_transport(state), _namespace="sample") as client:
    first = client.sync_history(SYMBOL, timeout=30, on_storage_error="raise")
    before_correction = datetime.now(UTC)
    repeated = client.sync_history(SYMBOL, full=True, timeout=30, on_storage_error="raise")
    assert repeated.summary["inserted_rows"] == 0
    assert repeated.summary["revised_rows"] == 0
    state["corrected"] = True
    corrected = client.sync_history(SYMBOL, full=True, timeout=30, on_storage_error="raise")
    current = client.load_stored_history(SYMBOL)
    previous = client.load_stored_history(SYMBOL, as_of=before_correction)
    versions = client.get_history_revisions(SYMBOL)
    status = client.history_status()
    assert previous.closing_price.iloc[-1] == 127
    assert current.closing_price.iloc[-1] == 128

## سلول ۵ — گزارش نتیجه

In [ ]:
display(pd.DataFrame([first.summary, repeated.summary, corrected.summary]))
display(status)
display(versions.tail(3))
print("Local observation-time selection only; not exchange publication-time reconstruction.")